In [ ]:
# ============================================================
# Controlled Tide Memory Experiment v4
# (Curriculum + Vector Gate + Gate Bias Init + SFA Decoupling)
#
# Key upgrades from v3:
#   1. Gate Bias Negative Init:
#      Initializes self.gate[-2].bias to -3.0 and weights to 0.0.
#      Starts gate at ~0.047, unlocking an initial retention flow
#      of (1 - gate) ≈ 0.953 across hundreds of steps (fixes the 0.5^T vanishing gradient).
#   2. Zero-out Default SFA Penalty:
#      Sets --lambda-state default to 0.0 to prevent adversarial
#      repression of persistent memory states.
#   3. Keeps the successful Curriculum Learning & Vector Channel Gating.
# ============================================================

import math
import json
import csv
import random
import argparse
import urllib.request
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# 1. Basic configuration
# ============================================================

TINY_SHAKESPEARE_URL = (
    "https://raw.githubusercontent.com/"
    "karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
)

CONTROL_CHARS = (
    "\n"
    "MEMASK"
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    "0123456789"
    " =?"
)

MEMORY_KEYS = list("ABCDEFGH")
MEMORY_VALUES = list("0123456789")


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ============================================================
# 2. Download Tiny Shakespeare
# ============================================================

def download_tiny_shakespeare(path):
    path = Path(path)
    if path.exists():
        print(f"[data] Using existing corpus: {path}")
        return

    path.parent.mkdir(parents=True, exist_ok=True)
    print("[data] Downloading Tiny Shakespeare...")
    urllib.request.urlretrieve(TINY_SHAKESPEARE_URL, path)
    print(f"[data] Saved to: {path}")


# ============================================================
# 3. Character vocabulary
# ============================================================

class CharVocab:
    def __init__(self, text):
        chars = sorted(set(text + CONTROL_CHARS))
        self.chars = chars
        self.stoi = {ch: i for i, ch in enumerate(chars)}
        self.itos = {i: ch for ch, i in self.stoi.items()}

    def encode(self, text):
        return [self.stoi[ch] for ch in text]

    def decode(self, ids):
        return "".join(self.itos[int(i)] for i in ids)

    def __len__(self):
        return len(self.chars)


# ============================================================
# 4. Normal LM batch
# ============================================================

def make_lm_batch(data, batch_size, seq_len, device):
    max_start = len(data) - seq_len - 1
    starts = torch.randint(0, max_start, (batch_size,))
    x = torch.stack([data[i:i + seq_len] for i in starts])
    y = torch.stack([data[i + 1:i + seq_len + 1] for i in starts])
    return x.to(device), y.to(device)


# ============================================================
# 5. Corpus span helper
# ============================================================

def sample_corpus_span(corpus, length, rng):
    if length <= 0:
        return ""
    start = rng.randint(0, len(corpus) - length - 1)
    return corpus[start:start + length]


# ============================================================
# 6. Synthetic memory example
# ============================================================

def make_memory_example(
    corpus,
    vocab,
    seq_len,
    min_distance,
    max_distance,
    rng,
    write_tide=0.30,
    preserve_tide=0.01,
    normal_tide=0.05
):
    key = rng.choice(MEMORY_KEYS)
    value = rng.choice(MEMORY_VALUES)

    prefix = f"MEM {key}={value}\n"
    query = f"\nASK {key}="

    total_length = seq_len + 1
    fixed_length = len(prefix) + len(query) + 1
    max_possible_distance = total_length - fixed_length

    if max_possible_distance < 1:
        raise ValueError("seq_len is too short for the memory task.")

    max_distance = min(max_distance, max_possible_distance)
    min_distance = min(min_distance, max_distance)

    distance = rng.randint(min_distance, max_distance)
    distractor = sample_corpus_span(corpus, distance, rng)

    answer_abs_pos = len(prefix) + len(distractor) + len(query)
    used_length = answer_abs_pos + 1
    suffix_length = total_length - used_length
    suffix = sample_corpus_span(corpus, suffix_length, rng)

    sequence = prefix + distractor + query + value + suffix
    assert len(sequence) == total_length

    ids = torch.tensor(vocab.encode(sequence), dtype=torch.long)
    x = ids[:-1]
    y = ids[1:]

    recall_position = answer_abs_pos - 1

    tide_target = torch.full((seq_len,), normal_tide, dtype=torch.float32)
    write_position = len(f"MEM {key}=")
    tide_target[write_position] = write_tide

    distractor_start = len(prefix)
    distractor_end = distractor_start + distance

    preserve_mask = torch.zeros(seq_len, dtype=torch.bool)
    if distance > 0:
        tide_target[distractor_start:distractor_end] = preserve_tide
        preserve_mask[distractor_start:distractor_end] = True

    query_position = recall_position

    return {
        "x": x,
        "y": y,
        "key": key,
        "value": value,
        "distance": distance,
        "recall_position": recall_position,
        "write_position": write_position,
        "query_position": query_position,
        "tide_target": tide_target,
        "preserve_mask": preserve_mask
    }


# ============================================================
# 7. Memory batch
# ============================================================

def make_memory_batch(
    corpus,
    vocab,
    batch_size,
    seq_len,
    min_distance,
    max_distance,
    device,
    rng,
    write_tide,
    preserve_tide,
    normal_tide
):
    examples = [
        make_memory_example(
            corpus=corpus,
            vocab=vocab,
            seq_len=seq_len,
            min_distance=min_distance,
            max_distance=max_distance,
            rng=rng,
            write_tide=write_tide,
            preserve_tide=preserve_tide,
            normal_tide=normal_tide
        )
        for _ in range(batch_size)
    ]

    x = torch.stack([e["x"] for e in examples])
    y = torch.stack([e["y"] for e in examples])
    recall_positions = torch.tensor([e["recall_position"] for e in examples])
    write_positions = torch.tensor([e["write_position"] for e in examples])
    query_positions = torch.tensor([e["query_position"] for e in examples])
    tide_targets = torch.stack([e["tide_target"] for e in examples])
    preserve_mask = torch.stack([e["preserve_mask"] for e in examples])

    return {
        "x": x.to(device),
        "y": y.to(device),
        "recall_positions": recall_positions.to(device),
        "write_positions": write_positions.to(device),
        "query_positions": query_positions.to(device),
        "tide_targets": tide_targets.to(device),
        "preserve_mask": preserve_mask.to(device)
    }


# ============================================================
# 8. GRU baseline
# ============================================================

class GRULM(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, tokens):
        x = self.embedding(tokens)
        states, _ = self.gru(x)
        logits = self.output(states)
        return {"logits": logits, "states": states}


# ============================================================
# 9. Tide Cell with Vector Channel Gating & Bias Initialization
# ============================================================

class TideMemoryCell(nn.Module):
    def __init__(self, embed_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim

        # Candidate state proposal
        self.proposal = nn.Sequential(
            nn.Linear(embed_dim + hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh()
        )

        # Vector Gate: Per-channel independent write gate
        self.gate = nn.Sequential(
            nn.Linear(embed_dim + hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, hidden_dim),
            nn.Sigmoid()
        )

        # ----------------------------------------------------
        # Gate Initialization:
        # Bias = -3.0 -> Sigmoid(-3.0) ≈ 0.047
        # Allows an initial retention coefficient of (1 - gate) ≈ 0.953
        # ----------------------------------------------------
        nn.init.zeros_(self.gate[-2].weight)
        nn.init.constant_(self.gate[-2].bias, -3.0)

    def forward(self, x, h):
        z = torch.cat([x, h], dim=-1)
        candidate = self.proposal(z)
        gate = self.gate(z)  # Shape: (B, hidden_dim)

        # State transition: per-channel selective update
        h_new = h + gate * (candidate - h)
        delta = h_new - h

        # Tide = RMS state change across the state vector
        tide = torch.sqrt(torch.mean(delta.pow(2), dim=-1) + 1e-8)

        return h_new, tide, gate


# ============================================================
# 10. Tide Language Model
# ============================================================

class TideLM(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.cell = TideMemoryCell(embed_dim, hidden_dim)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, tokens):
        B, T = tokens.shape
        h = torch.zeros(B, self.hidden_dim, device=tokens.device)

        logits_list = []
        states_list = []
        tides_list = []
        gates_list = []

        for t in range(T):
            x = self.embedding(tokens[:, t])
            h, tide, gate = self.cell(x, h)
            logits = self.output(h)

            logits_list.append(logits)
            states_list.append(h)
            tides_list.append(tide)
            gates_list.append(gate.mean(dim=-1))

        return {
            "logits": torch.stack(logits_list, dim=1),
            "states": torch.stack(states_list, dim=1),
            "tides": torch.stack(tides_list, dim=1),
            "gates": torch.stack(gates_list, dim=1)
        }


# ============================================================
# 11. Token CE & Recall Loss
# ============================================================

def token_cross_entropy(logits, targets):
    B, T, V = logits.shape
    return F.cross_entropy(
        logits.reshape(B * T, V),
        targets.reshape(B * T),
        reduction="none"
    ).reshape(B, T)


def compute_recall_loss(per_token_ce, recall_positions):
    batch_size = per_token_ce.shape[0]
    batch_index = torch.arange(batch_size, device=per_token_ce.device)
    recall_losses = per_token_ce[batch_index, recall_positions]
    return recall_losses.mean()


# ============================================================
# 12. SFA Anti-Collapse Regularization
# ============================================================

def state_regularization(states, target_variance=0.15):
    B, T, H = states.shape
    h = states.reshape(B * T, H)

    mean_loss = h.mean(dim=0).pow(2).mean()

    variance = h.var(dim=0, unbiased=False)
    variance_loss = (variance - target_variance).pow(2).mean()

    centered = h - h.mean(dim=0, keepdim=True)
    covariance = (centered.T @ centered) / max(centered.shape[0] - 1, 1)
    eye = torch.eye(H, device=states.device)
    off_diag = covariance * (1 - eye)
    covariance_loss = off_diag.pow(2).mean()

    return mean_loss + variance_loss + covariance_loss


# ============================================================
# 13. Metrics and Statistics
# ============================================================

def correlation(a, b):
    a = a.detach().reshape(-1)
    b = b.detach().reshape(-1)

    a = a - a.mean()
    b = b - b.mean()

    numerator = (a * b).mean()
    denominator = a.std(unbiased=False) * b.std(unbiased=False) + 1e-8
    return (numerator / denominator).item()


def tide_statistics(output, memory_batch):
    tides = output["tides"]
    B = tides.shape[0]
    batch_index = torch.arange(B, device=tides.device)

    write_tide = tides[batch_index, memory_batch["write_positions"]].mean().item()
    query_tide = tides[batch_index, memory_batch["query_positions"]].mean().item()

    preserve_values = tides[memory_batch["preserve_mask"]]
    preserve_tide = (
        preserve_values.mean().item()
        if preserve_values.numel() > 0
        else 0.0
    )

    return {
        "write_tide": write_tide,
        "preserve_tide": preserve_tide,
        "query_tide": query_tide
    }


# ============================================================
# 14. Training Objective
# ============================================================

def compute_training_loss(
    model_name,
    natural_output,
    natural_targets,
    memory_output,
    memory_batch,
    args
):
    natural_ce = token_cross_entropy(natural_output["logits"], natural_targets)
    natural_loss = natural_ce.mean()

    memory_ce = token_cross_entropy(memory_output["logits"], memory_batch["y"])
    memory_lm_loss = memory_ce.mean()

    recall_loss = compute_recall_loss(memory_ce, memory_batch["recall_positions"])

    task_loss = (
        args.normal_lm_weight * natural_loss
        + args.memory_lm_weight * memory_lm_loss
        + args.recall_weight * recall_loss
    )

    info = {
        "natural_lm": natural_loss.item(),
        "memory_lm": memory_lm_loss.item(),
        "recall_loss": recall_loss.item()
    }

    if model_name == "GRU":
        return task_loss, info

    total = task_loss
    if args.lambda_state > 0.0:
        reg_natural = state_regularization(natural_output["states"], args.target_variance)
        reg_memory = state_regularization(memory_output["states"], args.target_variance)
        state_reg = 0.5 * (reg_natural + reg_memory)
        total = total + args.lambda_state * state_reg

    tides_mem = memory_output["tides"]
    tide_stats = tide_statistics(memory_output, memory_batch)
    info.update(tide_stats)
    info["mean_tide"] = tides_mem.mean().item()

    if model_name == "TideNoLoss":
        info["tide_loss"] = 0.0
        info["tide_corr"] = 0.0
        return total, info

    if model_name == "AlwaysSlow":
        target_nat = torch.full_like(natural_output["tides"], args.preserve_tide)
        target_mem = torch.full_like(memory_output["tides"], args.preserve_tide)
    elif model_name == "ControlledTide":
        target_nat = torch.full_like(natural_output["tides"], args.normal_tide)
        target_mem = memory_batch["tide_targets"]
    else:
        raise ValueError(f"Unknown model: {model_name}")

    tide_loss_nat = F.mse_loss(natural_output["tides"], target_nat)
    tide_loss_mem = F.mse_loss(memory_output["tides"], target_mem)
    tide_loss = 0.5 * (tide_loss_nat + tide_loss_mem)

    total = total + args.lambda_tide * tide_loss
    tide_corr = correlation(tides_mem, target_mem)

    info["tide_loss"] = tide_loss.item()
    info["tide_corr"] = tide_corr

    return total, info


# ============================================================
# 15. Train Model (with Curriculum Schedule)
# ============================================================

def train_model(
    model,
    model_name,
    train_data,
    corpus,
    vocab,
    args,
    device
):
    print()
    print("=" * 72)
    print(f"Training: {model_name}")
    print("=" * 72)

    rng = random.Random(args.train_seed)
    torch.manual_seed(args.train_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.train_seed)

    model = model.to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=args.lr,
        weight_decay=args.weight_decay
    )

    model.train()

    for step in range(1, args.steps + 1):

        # ----------------------------------------------------
        # Dynamic Curriculum Learning Schedule
        # ----------------------------------------------------
        if step <= 300:
            curr_min_dist = 1
            curr_max_dist = 2
        elif step <= 600:
            progress = (step - 300) / 300.0
            curr_min_dist = int(1 + progress * (5 - 1))
            curr_max_dist = int(2 + progress * (40 - 2))
        else:
            progress = min(1.0, (step - 600) / max(1, (args.steps - 600)))
            curr_min_dist = int(5 + progress * (args.train_min_distance - 5))
            curr_max_dist = int(40 + progress * (args.train_max_distance - 40))

        curr_max_dist = max(curr_min_dist, curr_max_dist)

        # ----------------------------------------------------
        # Batches
        # ----------------------------------------------------
        x_nat, y_nat = make_lm_batch(
            train_data,
            args.batch_size,
            args.seq_len,
            device
        )

        mem = make_memory_batch(
            corpus=corpus,
            vocab=vocab,
            batch_size=args.batch_size,
            seq_len=args.seq_len,
            min_distance=curr_min_dist,
            max_distance=curr_max_dist,
            device=device,
            rng=rng,
            write_tide=args.write_tide,
            preserve_tide=args.preserve_tide,
            normal_tide=args.normal_tide
        )

        optimizer.zero_grad(set_to_none=True)

        natural_output = model(x_nat)
        memory_output = model(mem["x"])

        loss, info = compute_training_loss(
            model_name,
            natural_output,
            y_nat,
            memory_output,
            mem,
            args
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
        optimizer.step()

        if step == 1 or step % args.log_interval == 0:
            message = (
                f"[{model_name}] step={step:5d} "
                f"D=[{curr_min_dist:2d},{curr_max_dist:3d}] "
                f"loss={loss.item():.4f} "
                f"LM={info['natural_lm']:.3f} "
                f"MEM={info['memory_lm']:.3f} "
                f"RECALL={info['recall_loss']:.3f}"
            )

            if model_name != "GRU":
                message += (
                    f" | tide={info['mean_tide']:.3f}"
                    f" write={info['write_tide']:.3f}"
                    f" preserve={info['preserve_tide']:.3f}"
                    f" query={info['query_tide']:.3f}"
                )

            if model_name in ("AlwaysSlow", "ControlledTide"):
                message += f" corr={info['tide_corr']:.3f}"

            print(message)

    return model


# ============================================================
# 16. Evaluation and Prompting
# ============================================================

@torch.inference_mode()
def evaluate_language_model(model, val_data, batch_size, seq_len, device, batches=20):
    model.eval()
    losses = []

    for _ in range(batches):
        x, y = make_lm_batch(val_data, batch_size, seq_len, device)
        output = model(x)
        loss = token_cross_entropy(output["logits"], y).mean()
        losses.append(loss.item())

    mean_loss = sum(losses) / len(losses)
    perplexity = math.exp(min(mean_loss, 20))
    return mean_loss, perplexity


def make_recall_prompt(corpus, distance, rng):
    key = rng.choice(MEMORY_KEYS)
    value = rng.choice(MEMORY_VALUES)

    prefix = f"MEM {key}={value}\n"
    distractor = sample_corpus_span(corpus, distance, rng)
    query = f"\nASK {key}="

    prompt = prefix + distractor + query
    return {
        "prompt": prompt,
        "key": key,
        "value": value,
        "write_position": len(f"MEM {key}="),
        "distractor_start": len(prefix),
        "distractor_end": len(prefix) + distance,
        "query_position": len(prompt) - 1
    }


@torch.inference_mode()
def evaluate_recall(model, corpus, vocab, distances, samples, device, seed):
    model.eval()
    results = {}

    for distance in distances:
        rng = random.Random(seed + distance)
        prompts = []
        answers = []

        for _ in range(samples):
            example = make_recall_prompt(corpus, distance, rng)
            prompts.append(vocab.encode(example["prompt"]))
            answers.append(vocab.stoi[example["value"]])

        x = torch.tensor(prompts, dtype=torch.long, device=device)
        answers = torch.tensor(answers, dtype=torch.long, device=device)

        output = model(x)
        logits = output["logits"][:, -1, :]
        predictions = logits.argmax(dim=-1)

        accuracy = predictions.eq(answers).float().mean().item()
        results[str(distance)] = accuracy

        print(f"distance={distance:4d} accuracy={accuracy:.4f}")

    return results


# ============================================================
# 17. Visualization Utilities
# ============================================================

@torch.inference_mode()
def save_tide_trace(model, corpus, vocab, distance, device, output_dir, seed):
    if not isinstance(model, TideLM):
        return

    rng = random.Random(seed)
    example = make_recall_prompt(corpus, distance, rng)
    prompt = example["prompt"]

    ids = torch.tensor([vocab.encode(prompt)], dtype=torch.long, device=device)

    model.eval()
    output = model(ids)

    tides = output["tides"][0].cpu().tolist()
    gates = output["gates"][0].cpu().tolist()

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    csv_path = output_dir / "controlled_tide_trace.csv"
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["position", "character", "tide", "gate", "region"])

        for i, (ch, tide, gate) in enumerate(zip(prompt, tides, gates)):
            if i == example["write_position"]:
                region = "WRITE"
            elif example["distractor_start"] <= i < example["distractor_end"]:
                region = "PRESERVE"
            elif i == example["query_position"]:
                region = "QUERY"
            else:
                region = "NORMAL"

            writer.writerow([i, repr(ch), tide, gate, region])

    print(f"[saved] {csv_path}")

    try:
        import matplotlib.pyplot as plt
        plt.figure(figsize=(14, 5))
        plt.plot(range(len(tides)), tides, label="Actual Tide", color="#1f77b4")
        plt.axvline(example["write_position"], linestyle="--", color="red", label="WRITE")
        plt.axvline(example["query_position"], linestyle="--", color="green", label="QUERY")
        plt.xlabel("Character position")
        plt.ylabel("Tide magnitude")
        plt.title("Controlled Tide Memory Trace (Vector Gate)")
        plt.legend()
        plt.tight_layout()

        png_path = output_dir / "controlled_tide_trace.png"
        plt.savefig(png_path, dpi=160)
        plt.close()
        print(f"[saved] {png_path}")
    except ImportError:
        pass


def save_accuracy_plot(results, distances, output_dir):
    try:
        import matplotlib.pyplot as plt
        plt.figure(figsize=(10, 6))

        for name, values in results.items():
            y = [values[str(d)] for d in distances]
            plt.plot(distances, y, marker="o", label=name)

        plt.axhline(0.10, linestyle="--", color="gray", label="Random Baseline (10%)")
        plt.xlabel("Memory distance (characters)")
        plt.ylabel("Recall accuracy")
        plt.ylim(0, 1.05)
        plt.title("Memory Retention vs Distance (v4 Fixed)")
        plt.legend()
        plt.tight_layout()

        path = Path(output_dir) / "recall_accuracy_v4.png"
        plt.savefig(path, dpi=160)
        plt.close()
        print(f"[saved] {path}")
    except ImportError:
        pass


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def create_model(name, vocab_size, args):
    set_seed(args.model_seed)
    if name == "GRU":
        return GRULM(
            vocab_size=vocab_size,
            embed_dim=args.embed_dim,
            hidden_dim=args.hidden_dim
        )
    return TideLM(
        vocab_size=vocab_size,
        embed_dim=args.embed_dim,
        hidden_dim=args.hidden_dim
    )


# ============================================================
# 18. Main Pipeline
# ============================================================

def main(args):
    set_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[device] {device}")

    download_tiny_shakespeare(args.data)
    with open(args.data, "r", encoding="utf-8") as f:
        corpus = f.read()

    vocab = CharVocab(corpus)
    print(f"[data] characters: {len(corpus):,}")
    print(f"[data] vocabulary: {len(vocab)}")

    encoded = torch.tensor(vocab.encode(corpus), dtype=torch.long)
    split = int(len(encoded) * 0.9)
    train_data = encoded[:split]
    val_data = encoded[split:]

    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    model_names = ["GRU", "TideNoLoss", "AlwaysSlow", "ControlledTide"]
    models = {}

    for name in model_names:
        model = create_model(name, len(vocab), args)
        print()
        print(f"[{name}] parameters: {count_parameters(model):,}")

        model = train_model(
            model=model,
            model_name=name,
            train_data=train_data,
            corpus=corpus,
            vocab=vocab,
            args=args,
            device=device
        )
        models[name] = model
        torch.save(model.state_dict(), output_dir / f"{name}.pt")

    print()
    print("=" * 72)
    print("Natural language validation")
    print("=" * 72)

    validation = {}
    for name, model in models.items():
        loss, ppl = evaluate_language_model(
            model,
            val_data,
            args.batch_size,
            args.seq_len,
            device,
            args.validation_batches
        )
        validation[name] = {"loss": loss, "perplexity": ppl}
        print(f"{name:16s} loss={loss:.4f} ppl={ppl:.2f}")

    distances = [int(x) for x in args.eval_distances.split(",")]

    print()
    print("=" * 72)
    print("Long-range memory benchmark")
    print("=" * 72)
    print("Random baseline = 0.100")

    recall = {}
    for name, model in models.items():
        print()
        print(f"--- {name} ---")
        recall[name] = evaluate_recall(
            model=model,
            corpus=corpus,
            vocab=vocab,
            distances=distances,
            samples=args.eval_samples,
            device=device,
            seed=args.eval_seed
        )

    save_tide_trace(
        model=models["ControlledTide"],
        corpus=corpus,
        vocab=vocab,
        distance=args.trace_distance,
        device=device,
        output_dir=output_dir,
        seed=args.eval_seed + 999
    )

    results = {
        "config": vars(args),
        "validation": validation,
        "recall": recall
    }

    with open(output_dir / "results_v4.json", "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    save_accuracy_plot(recall, distances, output_dir)

    print()
    print("=" * 72)
    print("SUMMARY")
    print("=" * 72)
    print("Chance level: 0.100")

    for d in distances:
        line = f"D={d:4d}"
        for name in model_names:
            line += f" | {name}: {recall[name][str(d)]:.3f}"
        print(line)

    print()
    print("Experiment finished.")


# ============================================================
# 19. Argument Parsing
# ============================================================

if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="Controlled Tide Memory Experiment v4"
    )

    parser.add_argument("--data", type=str, default="data/input.txt")
    parser.add_argument("--output-dir", type=str, default="results_tide_v4")

    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--model-seed", type=int, default=123)
    parser.add_argument("--train-seed", type=int, default=456)
    parser.add_argument("--eval-seed", type=int, default=789)

    parser.add_argument("--embed-dim", type=int, default=64)
    parser.add_argument("--hidden-dim", type=int, default=128)

    parser.add_argument("--steps", type=int, default=1000)
    parser.add_argument("--batch-size", type=int, default=16)
    parser.add_argument("--seq-len", type=int, default=256)
    parser.add_argument("--lr", type=float, default=3e-4)
    parser.add_argument("--weight-decay", type=float, default=0.01)
    parser.add_argument("--grad-clip", type=float, default=1.0)
    parser.add_argument("--log-interval", type=int, default=100)

    parser.add_argument("--normal-lm-weight", type=float, default=0.5)
    parser.add_argument("--memory-lm-weight", type=float, default=0.5)
    parser.add_argument("--recall-weight", type=float, default=20.0)

    parser.add_argument("--train-min-distance", type=int, default=20)
    parser.add_argument("--train-max-distance", type=int, default=180)

    parser.add_argument("--write-tide", type=float, default=0.30)
    parser.add_argument("--preserve-tide", type=float, default=0.01)
    parser.add_argument("--normal-tide", type=float, default=0.05)

    parser.add_argument("--lambda-tide", type=float, default=1.0)
    # SFA penalty set to 0.0 by default to prevent adversarial repression of memory persistence
    parser.add_argument("--lambda-state", type=float, default=0.0)
    parser.add_argument("--target-variance", type=float, default=0.15)

    parser.add_argument(
        "--eval-distances",
        type=str,
        default="20,50,100,150,180,200,400,800"
    )
    parser.add_argument("--eval-samples", type=int, default=200)
    parser.add_argument("--validation-batches", type=int, default=20)
    parser.add_argument("--trace-distance", type=int, default=150)

    try:
        get_ipython()
        args = parser.parse_args(args=[])
    except NameError:
        args = parser.parse_args()

    main(args)

# Adaptive Tide Memory Experiment — V4

## 1. Experimental Configuration

| Parameter | Value |
|---|---|
| Device | CUDA |
| Dataset | `data\input.txt` |
| Characters | 1,115,394 |
| Vocabulary Size | 75 |
| Training Steps | 1,000 |
| Maximum Training Distance | 180 |
| Maximum Evaluation Distance | 800 |
| Random Baseline | 0.100 |

### Model Parameters

| Model | Parameters |
|---|---:|
| GRU | 88,971 |
| TideNoLoss | 76,363 |
| AlwaysSlow | 76,363 |
| ControlledTide | 76,363 |

---

## 2. Training Results

### 2.1 GRU

| Step | D Range | Total Loss | LM | MEM | RECALL |
|---:|:---:|---:|---:|---:|---:|
| 1 | [1, 2] | 90.2027 | 4.293 | 4.293 | 4.295 |
| 100 | [1, 2] | 53.0059 | 4.720 | 4.738 | 2.414 |
| 200 | [1, 2] | 50.3968 | 4.293 | 4.315 | 2.305 |
| 300 | [1, 2] | 49.5840 | 4.054 | 4.102 | 2.275 |
| 400 | [2, 14] | 48.8350 | 3.696 | 3.862 | 2.253 |
| 500 | [3, 27] | 40.2814 | 3.404 | 4.714 | 1.811 |
| 600 | [5, 40] | 39.0085 | 3.323 | 4.014 | 1.767 |
| 700 | [8, 75] | 25.3449 | 3.305 | 3.625 | 1.094 |
| 800 | [12, 110] | 17.3663 | 3.348 | 3.749 | 0.691 |
| 900 | [16, 145] | 17.4460 | 3.275 | 3.571 | 0.701 |
| 1000 | [20, 180] | 11.9619 | 3.220 | 3.531 | 0.429 |

### 2.2 TideNoLoss

| Step | D Range | Total Loss | LM | MEM | RECALL | Tide | Write | Preserve | Query |
|---:|:---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 1 | [1, 2] | 90.0763 | 4.321 | 4.323 | 4.288 | 0.005 | 0.005 | 0.005 | 0.005 |
| 100 | [1, 2] | 54.0540 | 6.446 | 6.459 | 2.380 | 0.007 | 0.081 | 0.075 | 0.012 |
| 200 | [1, 2] | 52.6897 | 6.082 | 6.012 | 2.332 | 0.006 | 0.084 | 0.072 | 0.010 |
| 300 | [1, 2] | 52.1357 | 5.752 | 5.716 | 2.320 | 0.006 | 0.070 | 0.081 | 0.010 |
| 400 | [2, 14] | 51.9436 | 5.434 | 5.492 | 2.324 | 0.019 | 0.066 | 0.048 | 0.016 |
| 500 | [3, 27] | 52.3684 | 5.211 | 5.300 | 2.356 | 0.020 | 0.052 | 0.048 | 0.074 |
| 600 | [5, 40] | 51.4571 | 4.457 | 4.833 | 2.341 | 0.037 | 0.060 | 0.066 | 0.066 |
| 700 | [8, 75] | 33.2164 | 3.880 | 6.139 | 1.410 | 0.081 | 0.224 | 0.078 | 0.103 |
| 800 | [12, 110] | 32.3169 | 3.583 | 5.110 | 1.399 | 0.144 | 0.265 | 0.163 | 0.243 |
| 900 | [16, 145] | 24.3169 | 3.582 | 4.529 | 1.013 | 0.161 | 0.333 | 0.168 | 0.307 |
| 1000 | [20, 180] | 16.0928 | 3.384 | 3.822 | 0.625 | 0.120 | 0.466 | 0.130 | 0.399 |

### 2.3 AlwaysSlow

| Step | D Range | Total Loss | LM | MEM | RECALL | Tide | Write | Preserve | Query | Corr |
|---:|:---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 1 | [1, 2] | 90.0763 | 4.321 | 4.323 | 4.288 | 0.005 | 0.005 | 0.005 | 0.005 | 0.000 |
| 100 | [1, 2] | 54.0543 | 6.446 | 6.459 | 2.380 | 0.007 | 0.081 | 0.075 | 0.012 | 0.000 |
| 200 | [1, 2] | 52.6900 | 6.082 | 6.012 | 2.332 | 0.006 | 0.084 | 0.072 | 0.010 | 0.000 |
| 300 | [1, 2] | 52.1361 | 5.752 | 5.716 | 2.320 | 0.006 | 0.070 | 0.081 | 0.010 | 0.000 |
| 400 | [2, 14] | 51.9455 | 5.433 | 5.491 | 2.324 | 0.019 | 0.066 | 0.048 | 0.017 | 0.000 |
| 500 | [3, 27] | 52.3667 | 5.210 | 5.299 | 2.356 | 0.020 | 0.052 | 0.048 | 0.074 | 0.000 |
| 600 | [5, 40] | 51.4631 | 4.460 | 4.839 | 2.341 | 0.036 | 0.060 | 0.065 | 0.065 | 0.000 |
| 700 | [8, 75] | 33.0302 | 3.799 | 5.487 | 1.419 | 0.073 | 0.225 | 0.079 | 0.123 | 0.000 |
| 800 | [12, 110] | 31.9308 | 3.601 | 4.994 | 1.381 | 0.134 | 0.269 | 0.151 | 0.248 | 0.000 |
| 900 | [16, 145] | 24.4410 | 3.506 | 4.469 | 1.022 | 0.141 | 0.328 | 0.147 | 0.321 | 0.000 |
| 1000 | [20, 180] | 16.0940 | 3.387 | 3.993 | 0.620 | 0.111 | 0.459 | 0.124 | 0.372 | 0.000 |

### 2.4 ControlledTide

| Step | D Range | Total Loss | LM | MEM | RECALL | Tide | Write | Preserve | Query | Corr |
|---:|:---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 1 | [1, 2] | 90.0785 | 4.321 | 4.323 | 4.288 | 0.005 | 0.005 | 0.005 | 0.005 | 0.047 |
| 100 | [1, 2] | 54.0562 | 6.446 | 6.459 | 2.380 | 0.007 | 0.081 | 0.075 | 0.012 | 0.199 |
| 200 | [1, 2] | 52.6919 | 6.082 | 6.012 | 2.332 | 0.007 | 0.084 | 0.072 | 0.010 | 0.208 |
| 300 | [1, 2] | 52.1387 | 5.752 | 5.716 | 2.320 | 0.006 | 0.070 | 0.081 | 0.010 | 0.164 |
| 400 | [2, 14] | 51.9560 | 5.428 | 5.486 | 2.325 | 0.018 | 0.066 | 0.049 | 0.021 | 0.032 |
| 500 | [3, 27] | 52.3429 | 5.200 | 5.289 | 2.355 | 0.023 | 0.052 | 0.049 | 0.076 | -0.041 |
| 600 | [5, 40] | 51.4415 | 4.451 | 4.855 | 2.339 | 0.038 | 0.060 | 0.067 | 0.062 | -0.110 |
| 700 | [8, 75] | 33.4920 | 3.904 | 6.007 | 1.427 | 0.083 | 0.220 | 0.080 | 0.112 | 0.145 |
| 800 | [12, 110] | 32.4320 | 3.594 | 5.060 | 1.405 | 0.148 | 0.271 | 0.160 | 0.241 | -0.001 |
| 900 | [16, 145] | 23.9958 | 3.537 | 4.526 | 0.998 | 0.151 | 0.343 | 0.162 | 0.309 | 0.019 |
| 1000 | [20, 180] | 16.0021 | 3.381 | 3.905 | 0.618 | 0.116 | 0.465 | 0.119 | 0.376 | 0.161 |

---

## 3. Natural Language Validation

| Model | Validation Loss | Perplexity (PPL) |
|---|---:|---:|
| **GRU** | **3.2844** | **26.69** |
| TideNoLoss | 3.4568 | 31.72 |
| AlwaysSlow | 3.4464 | 31.39 |
| ControlledTide | 3.4357 | 31.05 |

**Best validation performance:** GRU

---

## 4. Long-Range Memory Benchmark

Random baseline accuracy: **0.100**

| Distance | GRU | TideNoLoss | AlwaysSlow | ControlledTide |
|---:|---:|---:|---:|---:|
| 20 | **0.890** | 0.785 | 0.785 | 0.785 |
| 50 | **0.855** | 0.750 | 0.700 | 0.700 |
| 100 | **0.870** | 0.690 | 0.660 | 0.680 |
| 150 | **0.845** | 0.690 | 0.685 | 0.685 |
| 180 | **0.850** | 0.650 | 0.645 | 0.670 |
| 200 | **0.815** | 0.730 | 0.740 | 0.730 |
| 400 | **0.765** | 0.675 | 0.675 | 0.675 |
| 800 | **0.840** | 0.470 | 0.670 | 0.610 |

### Mean Recall Accuracy

| Model | Mean Accuracy |
|---|---:|
| **GRU** | **0.8413** |
| TideNoLoss | 0.6800 |
| AlwaysSlow | 0.6944 |
| ControlledTide | 0.6913 |

---

## 5. Experimental Observations

### 5.1 Language Modeling

- GRU achieved the lowest validation loss (3.2844).
- GRU also achieved the lowest perplexity (26.69).
- ControlledTide performed best among the three Tide variants in language modeling.
- All Tide variants showed higher validation perplexity than GRU.

### 5.2 Long-Range Memory

- GRU achieved the highest recall accuracy at every tested distance.
- At distance 800:
  - GRU: 84.0%
  - TideNoLoss: 47.0%
  - AlwaysSlow: 67.0%
  - ControlledTide: 61.0%
- AlwaysSlow achieved the highest average recall accuracy among the Tide variants.
- ControlledTide did not consistently outperform AlwaysSlow.

### 5.3 Tide Dynamics

At training step 1000:

| Metric | TideNoLoss | AlwaysSlow | ControlledTide |
|---|---:|---:|---:|
| Tide | 0.120 | 0.111 | 0.116 |
| Write | 0.466 | 0.459 | 0.465 |
| Preserve | 0.130 | 0.124 | 0.119 |
| Query | 0.399 | 0.372 | 0.376 |
| Corr | — | 0.000 | 0.161 |

The ControlledTide correlation signal fluctuated during training,
with the final logged value reaching 0.161.

---

## 6. Conclusion

Under the current experimental configuration:

1. GRU outperformed all Tide variants in both language modeling and long-range recall.
2. ControlledTide achieved the best validation perplexity among Tide variants.
3. AlwaysSlow achieved the highest mean recall accuracy among Tide variants.
4. The controlled tide mechanism has not yet demonstrated a consistent performance advantage.
5. Further experiments are required to determine whether tide control improves memory retention under different training conditions.

**Overall result:** The current Tide architecture does not outperform the GRU baseline.

---

## 7. Saved Experiment Artifacts

| File | Description |
|---|---|
| `results_tide_v4/controlled_tide_trace.csv` | ControlledTide training trace |
| `results_tide_v4/controlled_tide_trace.png` | Tide dynamics visualization |
| `results_tide_v4/recall_accuracy_v4.png` | Recall accuracy comparison |

**Experiment Status:** Finished